# Time series for website
***

This script combines the daily discharge records with the daily basin meteorology computed from the EMO-1 dataset and exports a time series per gauging station to be plotted in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

from ocab.config import Config
from ocab.plots.reservoirs import plot_reservoir_timeseries, create_reservoir_html

## Configuration

In [2]:
cfg = Config('config_BEAVERS_v100.yml')

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_out = Path('../../docs/timeseries/reservoirs')
path_plots = path_out / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# decimals in output timeseries
rounding = {
    'storage_mcm': 3,
    'filling': 3,
    'inflow_cms': 3,
    'inflow_mm': 1,
    'outflow_cms': 3,
    'outflow_mm': 1,
    'temp_degC': 1,
    'precip_mm': 1,
    'pet_mm': 1,
}

variables = {
    'storage': 'storage_mcm',
    'inflow': 'inflow_cms',
    'outflow': 'outflow_cms',
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}


## Create time series


In [ ]:
# load dams
dams = gpd.read_file(cfg.path_gis / 'dams.geojson').set_index('id')

# process timeseries for each station
for ID in tqdm(dams.index, desc='stations'):
# for ID in tqdm([8001, 8031], desc='stations'):
    
    # reservoir operations
    try:
        resops = pd.read_parquet(path_in / 'resops' / f'{ID:04d}.parquet')
        resops.rename(columns=variables, inplace=True, errors='ignore')
        # compute reservoir filling
        resops['filling'] = resops['storage_mcm'] / dams.loc[ID, 'cap_mcm']
        # compute specific discharge (mm/day)
        for flow in ['inflow', 'outflow']:
            var = f'{flow}_cms'
            if var in resops.columns:
                resops[f'{flow}_mm'] = resops[var] / dams.loc[ID, 'catch_skm'] * 86400 / 1000
    except Exception as e:
        print(f'Error loading discharge timeseries for station {ID:04d}: {e}')
        continue

    # meteo timeseries
    try:
        meteo = pd.read_parquet(path_in / 'meteo' / 'EMO1' / f'{ID:04d}.parquet').loc[ID]
        meteo.rename(columns=variables, inplace=True, errors='ignore')
        # correct dates
        meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
        start = max(meteo.first_valid_index(), resops.first_valid_index())
        end = min(meteo.last_valid_index(), resops.last_valid_index())
        meteo = meteo.loc[start:end]
    except Exception as e:
        print(f'Error loading meteo timeseries for station {ID:04d}: {e}')
        break
        continue

    # merge timeseries
    ts = pd.concat([resops, meteo], axis=1)
    ts = ts[ts.columns.intersection(rounding)].round(rounding)
    
    # # export timeseries
    # ts.to_parquet(path_out / f'{ID:04d}.parquet')

    # extract attributes and time series
    attrs = dams.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title(), 
            attrs['river'].title(), 
            attrs['basin'].title()
        )
        
        fig = plot_reservoir_timeseries(
            ts,
            attrs,
            title=title,
            save=True
        )

        # save plot as HTML
        create_reservoir_html(
            fig, 
            path=path_plots / f'{ID}.html', 
            start=ts.index.min().strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except:
        print(f"The plot for time series {ID} couldn't be created")

stations:   0%|          | 0/337 [00:00<?, ?it/s]

Empty time series


The plot for time series 8012 couldn't be created
The plot for time series 9824 couldn't be created
The plot for time series 9844 couldn't be created
The plot for time series 9867 couldn't be created
The plot for time series 9877 couldn't be created
The plot for time series 9881 couldn't be created
The plot for time series 9895 couldn't be created
